In [2]:
import numpy as np
import pandas as pd
import time
import os
import random
import joblib

import elkai
import tsplib95
from scipy.spatial.distance import cdist
from scipy.stats import skew, kurtosis
from scipy.sparse.csgraph import minimum_spanning_tree

from mealpy import GA, SA, PSO, ACOR
from mealpy.utils.problem import Problem
from mealpy.utils.space import FloatVar

class TSPProblem(Problem):
    def __init__(self, D, **kwargs):
        self.D = D
        self.n = len(D)
        bounds = FloatVar(lb=[0.0] * self.n, ub=[self.n - 1.0] * self.n)
        super().__init__(
            bounds=bounds,
            minmax="min",
            **kwargs
        )

    def obj_func(self,x):
        tour = np.argsort(x).tolist()
        cost = sum(self.D[tour[i]][tour[i+1]] for i in range(self.n - 1))
        cost += self.D[tour[-1]][tour[0]]
        return cost

def two_opt(tour, D):
    n = len(tour)
    improved = True
    while improved:
        improved = False
        for i in range(1, n-1):
            for j in range(i+1, n):
                a, b = tour[i-1], tour[i]
                c, d = tour[j], tour[(j+1) % n]
                if D[a][b] + D[c][d] > D[a][c] + D[b][d]:
                    tour[i:j+1] = tour[i:j+1][::-1]
                    improved = True
    return tour


In [3]:
def run_mealpy(model,D):
    problem = TSPProblem(D=D, log_to=None)
    t0 = time.time()
    model.solve(problem)
    runtime = time.time() - t0
    best_x = model.g_best.solution
    tour = np.argsort(best_x).tolist()
    tour = two_opt(tour, D)
    cost = sum(D[tour[i]][tour[i+1]] for i in range(len(tour)-1)) + D[tour[-1]][tour[0]]
    history = model.history.list_global_best_fit
    return cost, runtime, history

def run_lk(D):
    D_int = (D * 1000).astype(int).tolist()
    t0 = time.time()
    tour = elkai.solve_int_matrix(D_int)
    runtime = time.time() - t0
    cost = sum(D[tour[i]][tour[i+1]] for i in range(len(tour) - 1)) + D[tour[-1]][tour[0]]
    return cost, runtime, None

In [4]:
def get_algo_pool(n):
    if n<=30:
        epochs, pop = 200,100
    elif n<=75:
        epochs, pop = 300,100
    elif n<=100:
        epochs, pop = 400,100
    else:
        epochs, pop = 500,150
        
    return {
        "OGA": lambda D, e=epochs, p=pop: run_mealpy(GA.OriginalGA(epoch=e, pop_size=p), D),
        "SA":  lambda D, e=epochs: run_mealpy(SA.OriginalSA(epoch=e, pop_size=10), D),
        "LK":  lambda D: run_lk(D),
        "PSO": lambda D, e=epochs, p=pop: run_mealpy(PSO.OriginalPSO(epoch=e, pop_size=p), D),
        #"ACO": lambda D, e=epochs, p=pop: run_mealpy(ACOR.OriginalACOR(epoch=e, pop_size=p), D)
        
    }


In [5]:
def compute_distance_matrix(coords, distance_type="euclidean"):
    n = len(coords)
    D = np.zeros((n,n))

    if distance_type == "euclidean":
        D = cdist(coords, coords, metric='euclidean')

    elif distance_type == "manhattan":
        D = cdist(coords, coords, metric="cityblock")

    elif distance_type == "chebyshev":
        D = cdist(coords, coords, metric="chebyshev")

    elif distance_type == "ceil_euclidean":
        D = np.ceil(cdist(coords, coords, metric="euclidean"))

    else:
        raise ValueError(f"Unsupported distance type: {distance_type}")
        
    np.fill_diagonal(D, 0)
    return D

DISTANCE_TYPE_ENCODING = {
    "euclidean": 0,
    "manhattan": 1,
    "chebyshev": 2,
    "ceil_euclidean": 3
}

In [6]:
def generate_instance(n, distribution, seed, distance_type="euclidean", noise_factor=0.0, instance_type="standard"):
    rng = np.random.default_rng(seed)

    if instance_type == "random_cost":
        D = rng.uniform(100, 1000, (n,n))
        D = (D+D.T)/2
        np.fill_diagonal(D,0)
        coords = None
        return coords, D

    elif instance_type == "triangle_violation":
        coords = rng.uniform(0,1000,(n,2))
        D = cdist(coords, coords, metric="euclidean")
        for _ in range (int(n*n*0.3)):
            i, j, k = rng.choice(n,3, replace=False)
            if D[i][j] <= D[i][j] + D[k][j]:
                D[i][j] = (D[i][k] + D[k][j]) * rng.uniform(1.1, 1.5)
                D[j][i] = D[i][j]
        np.fill_diagonal(D,0)
        return coords, D

    if distribution == "uniform":
        coords = rng.uniform(0, 1000, (n,2))

    elif distribution == "clustered":
        centers = rng.uniform(100, 900, (max(2, n//10),2))
        coords = np.array([centers[i % max(2, n//10)] + rng.normal(0,30,2) for i in range(n)])

    elif distribution == "grid":
        side = int(np.ceil(np.sqrt(n)))
        coords = np.array([[x,y] for x in range(side) for y in range(side)])[:n] * (1000/side)

    elif distribution == "diagonal":
        t = rng.uniform(0, 1000, n)
        noise = rng.normal(0,20,(n,2))
        coords = np.column_stack([t, t]) + noise

    D = compute_distance_matrix(coords, distance_type)
    
    if noise_factor > 0.0:
        noise = rng.uniform(0,noise_factor,(n,n))
        noise = (noise + noise.T)/2
        np.fill_diagonal(noise, 0)
        D = D + noise * D.mean()
        np.fill_diagonal(D,0)

    return coords, D

INSTANCE_TYPE = {
    "standard": 0,
    "random_cost": 1,
    "triangle_violation": 2
}

In [7]:
def load_tsplib(name, folder="./tsplib"):
    problem = tsplib95.load(os.path.join(folder, f"{name}.tsp"))
    n = problem.dimension
    nodes = list(problem.get_nodes())

    D = np.zeros((n, n))
    for i in nodes:
        for j in nodes:
            D[i-1][j-1] = problem.get_weight(i, j)

    coords = None
    if problem.node_coords:
        coords = np.array([problem.node_coords[i] for i in nodes])

    return coords, D

In [8]:
def extract_features(coords, D, distance_type, noise_factor, instance_type):
    n = len(D)
    upper = D[np.triu_indices(n, k=1)]

    D_nn = D.copy()
    np.fill_diagonal(D_nn, np.inf)
    nn = D_nn.min(axis=1)

    mst_arr = minimum_spanning_tree(D).toarray()
    mst_vals = mst_arr[mst_arr > 0]
    
    violations = 0
    checks = 0
    for i in range(min(n,20)):
        for j in range(min(n,20)):
            for k in range(min(n,20)):
                if i!=j and j!=k and i!=k:
                    if D[i][j] > D[i][k] + D[k][j]:
                        violations += 1
                    checks += 1

    features = {
        "n_cities": n,
        "noise_factor": noise_factor,
        "distance_type": distance_type,
        "triangle_violation_rate": violations / checks if checks > 0 else 0,
        "instance_type": INSTANCE_TYPE.get(instance_type, -1),
        "avg_edge": upper.mean(),
        "std_edge": upper.std(),
        "cv_edge": upper.std() / upper.mean(),
        "skew_edge": float(skew(upper)),
        "kurtosis_edge": float(kurtosis(upper)),
        "min_edge": upper.min(),
        "max_edge": upper.max(),

        "avg_nn": nn.mean(),
        "std_nn": nn.std(),
        "cv_nn": nn.std() / nn.mean(),
        "skew_nn": float(skew(nn)),

        "mst_cost": mst_vals.sum(),
        "avg_mst_edge": mst_vals.mean(),
        "std_mst_edge": mst_vals.std(),
        "mst_nn_ratio": mst_vals.mean() / nn.mean()
    }

    if coords is not None:
        centroid = coords.mean(axis=0)
        centroid_dists = np.linalg.norm(coords - centroid, axis=1)

        x_range = coords[:, 0].max() - coords[:, 0].min()
        y_range = coords[:, 1].max() - coords[:, 1].min()

        features.update({
            "avg_centroid_dist": centroid_dists.mean(),
            "std_centroid_dist": centroid_dists.std(),
            "cv_centroid_dist": centroid_dists.std() / centroid_dists.mean(),
            "bbox_area": x_range * y_range,
            "bbox_ratio": x_range / (y_range +1e-9)
        })

    else:
        for col in ["avg_centroid_dist","std_centroid_dist",
                    "cv_centroid_dist","bbox_area","bbox_ratio"]:
            features[col] = np.nan
        
    return features


In [9]:
def solve_concorde(coords):
    try:
        from concorde.tsp import TSPSolver
        solver = TSPSolver.from_data(
            coords[:, 0].tolist(),
            coords[:, 1].tolist(),
            norm = "EUC_2D"
        )
        solution = solver.solve(verbose=False)
        tour = list(solution.tour)
        D = cdist(coords, coords)
        cost = sum(D[tour[i]][tour[i+1]] for i in range(len(tour)-1)) + D[tour[-1]][tour[0]]
        
        return cost
    except ImportError:
        return None

In [10]:
def run_all(D, coords=None, n_runs=3, use_concorde=False):
    n = len(D)
    results = {}
    pool = get_algo_pool(n)

    for name, algo_fn  in pool.items():
        run_costs = []
        run_times = []

        for _ in range(n_runs):
            cost, runtime, _ = algo_fn(D)
            run_costs.append(cost)
            run_times.append(runtime)
        results[name] = {
            "best_cost": min(run_costs),
            "avg_cost": np.mean(run_costs),
            "std_cost": np.std(run_costs),
            "avg_time": np.mean(run_times),
        }

    optimal = None
    if use_concorde and coords is not None and len(D) <= 150:
        optimal = solve_concorde(coords)

    return results, optimal

In [11]:
def validate_tour(tour, n):
    if len(tour)!=n:
        return False, f"Tour length {len(tour)} does not match n={n}"

    expected = set(range(n))
    actual = set(tour)

    if actual!=expected:
        missing = expected - actual
        extra = actual - expected
        msh = ""
        if missing:
            msg += f"Missing cities: {missing}"
        if extra:
            msg += f"Invalid cities: {extra}"
        return False, msg
    if len(tour) != len(set(tour)):
        duplicates = [c for c in tour if tour.count(c) > 1]
        return False, f"Duplicate cities: {set(duplicates)}"
    
    return True, "Valid"

In [12]:
def build_row(coords, D, source, label, n_runs=3, use_concorde=False, 
              distance_type="euclidean", noise_factor=0.0, instance_type="standard"):
    features = extract_features(coords, D,
                               distance_type=distance_type,
                               noise_factor=noise_factor,
                               instance_type=instance_type)
    algo_results, optimal = run_all(
        D, coords, n_runs=n_runs, use_concorde=use_concorde
    )

    best_algo = min(algo_results, key=lambda k: algo_results[k]["best_cost"])

    row = {
        **features,
        "source": source,
        "label": label,
        "best_algo": best_algo,
        "optimal": optimal,
    }

    for algo, metrics in algo_results.items():
        for metric, val in metrics.items():
            row[f"{algo}_{metric}"] = val

        if optimal:
            row[f"{algo}_gap"] = (
                (algo_results[algo]["best_cost"] - optimal / optimal)
            )
        return row
    
def build_dataset(configs, source="generated", use_concorde=False):
    rows=[]
    for i, cfg in enumerate(configs):
        if source == "generated":
            coords, D = generate_instance(
                cfg["n"], cfg["distribution"], cfg["seed"], distance_type=cfg.get("distance_type", "euclidean"),
                noise_factor=cfg.get("noise_factor",0.0), instance_type=cfg.get("instance_type", "standard")
            )
            label = (f"{cfg['distribution']}_{cfg['n']}_{cfg['seed']}_"
                    f"{cfg.get('distance_type','euclidean')}_"
                    f"{cfg.get('instance_type', 'standard')}")
        else:
            coords, D = load_tsplib(cfg["name"])
            label = cfg["name"]
        
        row = build_row(
            coords, D,
            source=source,
            label=label,
            use_concorde=use_concorde,
            distance_type=cfg.get("distance_type", "euclidean"),
            noise_factor=cfg.get('noise_factor',0.0),
            instance_type=cfg.get("instance_type", "standard")
        )
        rows.append(row)
        print(f"[{i+1}/{len(configs)}] {label} -> best: {row['best_algo']}")
    
    return pd.DataFrame(rows)



# Validation


In [16]:
def run_mealpy_val(model,D):
    problem = TSPProblem(D=D, log_to=None)
    t0 = time.time()
    model.solve(problem)
    runtime = time.time() - t0
    best_x = model.g_best.solution
    tour = np.argsort(best_x).tolist()
    tour = two_opt(tour, D)
    cost = sum(D[tour[i]][tour[i+1]] for i in range(len(tour)-1)) + D[tour[-1]][tour[0]]
    history = model.history.list_global_best_fit
    return cost, runtime, history, tour

def run_lk_val(D):
    D_int = (D * 1000).astype(int).tolist()
    t0 = time.time()
    tour = elkai.solve_int_matrix(D_int)
    runtime = time.time() - t0
    cost = sum(D[tour[i]][tour[i+1]] for i in range(len(tour) - 1)) + D[tour[-1]][tour[0]]
    return cost, runtime, None, tour

def get_algo_pool_val(n):
    if n<=30:
        epochs, pop = 200,100
    elif n<=75:
        epochs, pop = 300,75
    elif n<=100:
        epochs, pop = 400,100
    elif n<=150:
        epochs, pop = 500,150
    else:
        epochs, pop = 800,200
        
    return {
        "OGA": lambda D, e=epochs, p=pop: run_mealpy_val(GA.OriginalGA(epoch=e, pop_size=p), D),
        "SA":  lambda D, e=epochs: run_mealpy_val(SA.OriginalSA(epoch=e, pop_size=10), D),
        "LK":  lambda D: run_lk_val(D),
        "PSO": lambda D, e=epochs, p=pop: run_mealpy_val(PSO.OriginalPSO(epoch=e, pop_size=p), D),
        "ACO": lambda D, e=epochs, p=pop: run_mealpy_val(ACOR.OriginalACOR(epoch=e, pop_size=p), D),
    }
def test_solvers(n=20, distribution='uniform',seed=0, tsplib_name=None):
    if tsplib_name is not None:
        coords, D = coords, D = load_tsplib(tsplib_name)
        n = len(D)
        label = tsplib_name
    else:
        coords, D = generate_instance(n, distribution, seed, distance_type="manhattan", noise_factor=0.1, instance_type="triangle_violation")
    pool = get_algo_pool_val(n)

    print(f"\nTesting  on {distribution} instance, n={n}")
    print('-'*50)

    for name, algo_fn in pool.items():
        cost, runtime, _, tour = algo_fn(D)
        is_valid, msg = validate_tour(tour,n)
        print(f"{name:6} | valid={is_valid} | cost={cost:.2f} | runtime={runtime}| {msg}")

TSPLIB_PROBLEMS = [
    "eil51", "berlin52", "st70", "eil76",
    "pr76", "kroA100", "eil101", "lin105"
]
test_solvers(n=20, distribution="uniform", seed=0)
test_solvers(n=20, distribution="clustered", seed=0)
test_solvers(n=20, distribution="grid", seed=0)
test_solvers(n=50, distribution="diagonal", seed=0)
test_solvers(n=70, distribution="clustered", seed=0)

"""test_solvers(tsplib_name="eil51")
test_solvers(tsplib_name="berlin52")
test_solvers(tsplib_name="st70")
test_solvers(tsplib_name="bays29")"""


Testing  on uniform instance, n=20
--------------------------------------------------
OGA    | valid=True | cost=6236.15 | runtime=0.030297517776489258| Valid
SA     | valid=True | cost=6808.60 | runtime=0.018456220626831055| Valid
LK     | valid=True | cost=5721.06 | runtime=0.026153564453125| Valid
PSO    | valid=True | cost=6368.44 | runtime=0.5902154445648193| Valid
ACO    | valid=True | cost=6368.63 | runtime=2.7370779514312744| Valid

Testing  on clustered instance, n=20
--------------------------------------------------
OGA    | valid=True | cost=6412.57 | runtime=0.020182371139526367| Valid
SA     | valid=True | cost=6776.99 | runtime=0.014512300491333008| Valid
LK     | valid=True | cost=5721.06 | runtime=0.02431631088256836| Valid
PSO    | valid=True | cost=5780.41 | runtime=0.584303617477417| Valid
ACO    | valid=True | cost=6549.71 | runtime=2.6805524826049805| Valid

Testing  on grid instance, n=20
--------------------------------------------------
OGA    | valid=True | c

'test_solvers(tsplib_name="eil51")\ntest_solvers(tsplib_name="berlin52")\ntest_solvers(tsplib_name="st70")\ntest_solvers(tsplib_name="bays29")'

In [ ]:
train_configs = [
    {"n": n, "distribution": d, "seed": s, "distance_type": dt, "noise_factor": nf, "instance_type": it}
    for n in [10,20,50,75,100,130]
    for d in ["uniform","clustered", "grid","diagonal"]
    for s in range(15)
    for dt in ["euclidean","manhattan","ceil_euclidean","chebyshev"]
    for nf in [0.1, 0.2]
    for it in ["random_cost","triangle_violation"]
]

TSPLIB_PROBLEMS = [
    "eil51", "berlin52", "st70", "eil76",
    "pr76", "kroA100", "eil101", "lin105"
]
val_configs = [{"name": p} for p in TSPLIB_PROBLEMS]
train_df = build_dataset(train_configs, source="generated", use_concorde=False)
val_df = build_dataset(val_configs, source="tsplib", use_concorde=True)

train_df.to_csv("train_meta.csv", index=False)
val_df.to_csv("val_meta.csv", index=False)

[1/5760] uniform_10_0_euclidean_random_cost -> best: SA
[2/5760] uniform_10_0_euclidean_triangle_violation -> best: LK
[3/5760] uniform_10_0_euclidean_random_cost -> best: PSO
[4/5760] uniform_10_0_euclidean_triangle_violation -> best: LK
[5/5760] uniform_10_0_manhattan_random_cost -> best: PSO
[6/5760] uniform_10_0_manhattan_triangle_violation -> best: LK
[7/5760] uniform_10_0_manhattan_random_cost -> best: OGA
[8/5760] uniform_10_0_manhattan_triangle_violation -> best: SA
[9/5760] uniform_10_0_ceil_euclidean_random_cost -> best: SA
[10/5760] uniform_10_0_ceil_euclidean_triangle_violation -> best: OGA
[11/5760] uniform_10_0_ceil_euclidean_random_cost -> best: PSO
[12/5760] uniform_10_0_ceil_euclidean_triangle_violation -> best: OGA
[13/5760] uniform_10_0_chebyshev_random_cost -> best: SA
[14/5760] uniform_10_0_chebyshev_triangle_violation -> best: OGA
[15/5760] uniform_10_0_chebyshev_random_cost -> best: PSO
[16/5760] uniform_10_0_chebyshev_triangle_violation -> best: LK
[17/5760] uni